In [ ]:
import fitz  # PyMuPDF
import re
import os
import sys

# Make src/ importable from notebook
sys.path.insert(0, os.path.abspath(".."))

from src.ingestion.pdf_loader import load_pdf
from src.ingestion.text_cleaner import clean_text

In [ ]:
PDF_PATH = "E:\\Project\\llm_data_factory\\reports and research paper\\Minor_Project_II (2).pdf"  # ← change this

# Basic file info
print(f"File exists : {os.path.exists(PDF_PATH)}")
print(f"File size   : {os.path.getsize(PDF_PATH) / 1024:.1f} KB")

File exists : True
File size   : 2344.3 KB


In [ ]:
doc = fitz.open(PDF_PATH)

print(f"Total pages : {len(doc)}")
print(f"Metadata    : {doc.metadata}")
print(f"\nTable of Contents:")
toc = doc.get_toc()
if toc:
    for item in toc:
        level, title, page = item
        indent = "  " * (level - 1)
        print(f"{indent}• {title} (page {page})")
else:
    print("  No table of contents found.")

Total pages : 34
Metadata    : {'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.27', 'creationDate': 'D:20260405053103Z', 'modDate': 'D:20260405053103Z', 'trapped': '', 'encryption': None}

Table of Contents:
• Declaration by the Candidate(s) (page 2)
• Approval Certificate (page 3)
• Certificate by the Examiners (page 4)
• Acknowledgement (page 5)
• Abstract (page 6)
• Table of Contents (page 6)
• List of Figures (page 6)
• List of Tables (page 6)
• Introduction (page 12)
  • Core Problem: The Assumption of Pristine Context (page 12)
  • Motivation: Architectural Robustification (page 12)
  • The Proposed Solution: Curriculum Noise Injection (page 12)
  • Primary Objectives (page 13)
  • Organization of the Report (page 13)
• Literature Review (page 14)
  • The Baseline: Achieving Zero-Shot Logic on Clean Data (page 14)
  • Discovering the Flaw: The Distractibility Phenomenon (page 14)
  • Early 

In [ ]:
print("=" * 60)
for page_num in range(min(3, len(doc))):
    page = doc[page_num]
    text = page.get_text("text")
    print(f"\n{'='*60}")
    print(f"PAGE {page_num + 1} — {len(text)} characters")
    print(f"{'='*60}")
    print(text[:1000])   # Show first 1000 chars of each page
    print("...")


PAGE 1 — 594 characters
A
Project Report
on
Robustifying Language Models with
Curriculum-Based Noise Injection
Submitted to
BHILAI INSTITUTE OF TECHNOLOGY, DURG
Affiliated to
CHHATTISGARH SWAMI VIVEKANAND TECHNICAL
UNIVERSITY
BHILAI
in partial fulfillment of
Bachelor of Technology
in
Computer Science and Engineering
by
Ayush Pradhan
∥
6th Semester ∥
300111323013
Aaryan Sahu
∥
6th Semester ∥
300111323001
Under the Guidance of
Prof. Saurabh Singh
Assistant Professor
Department of Computer Science and Engineering
Bhilai Institute of Technology,
Bhilai House, GE Road, Durg, Chhattisgarh 491001
Session:
2025 – 2026

...

PAGE 2 — 896 characters
DECLARATION BY THE CANDIDATE(s)
We the undersigned, solemnly declare that the report of the project work entitled “Robustify-
ing Language Models with Curriculum-Based Noise Injection”, is based on our own work
carried out during the course of our study under the supervision of Prof. Saurabh Singh,
Assistant Professor(Department of Computer Science 

In [ ]:
raw_text = load_pdf(PDF_PATH)

# Count non-ASCII characters (garbage symbols from scanned PDFs)
non_ascii = [ch for ch in raw_text if ord(ch) > 127]
print(f"Total characters     : {len(raw_text)}")
print(f"Non-ASCII characters : {len(non_ascii)}")
print(f"Non-ASCII sample     : {set(non_ascii[:50])}")

# Count blank lines
blank_lines = len(re.findall(r'\n\s*\n', raw_text))
print(f"\nBlank line groups    : {blank_lines}")

# Count standalone numbers (likely page numbers)
page_numbers = re.findall(r'^\s*\d+\s*$', raw_text, flags=re.MULTILINE)
print(f"Standalone numbers   : {len(page_numbers)}")
print(f"Sample               : {page_numbers[:10]}")

[pdf_loader] Extracted 52877 characters from E:\Project\llm_data_factory\reports and research paper\Minor_Project_II (2).pdf
Total characters     : 52877
Non-ASCII characters : 179
Non-ASCII sample     : {'“', '∥', '≥', '’', '”', '•', '—', '–'}

Blank line groups    : 68
Standalone numbers   : 102
Sample               : ['300111323013', '300111323001', '13', '01', '1', '1', '1', '1', '1', '2']


In [ ]:
cleaned_text = clean_text(raw_text)

print("BEFORE CLEANING")
print("-" * 40)
print(raw_text[:500])

print("\n\nAFTER CLEANING")
print("-" * 40)
print(cleaned_text[:500])

print(f"\n\nOriginal length : {len(raw_text)}")
print(f"Cleaned length  : {len(cleaned_text)}")
print(f"Removed         : {len(raw_text) - len(cleaned_text)} characters")

[text_cleaner] Cleaned text length: 51881 characters
BEFORE CLEANING
----------------------------------------


--- PAGE 1 ---

A
Project Report
on
Robustifying Language Models with
Curriculum-Based Noise Injection
Submitted to
BHILAI INSTITUTE OF TECHNOLOGY, DURG
Affiliated to
CHHATTISGARH SWAMI VIVEKANAND TECHNICAL
UNIVERSITY
BHILAI
in partial fulfillment of
Bachelor of Technology
in
Computer Science and Engineering
by
Ayush Pradhan
∥
6th Semester ∥
300111323013
Aaryan Sahu
∥
6th Semester ∥
300111323001
Under the Guidance of
Prof. Saurabh Singh
Assistant Professor
Department of Computer Science and En


AFTER CLEANING
----------------------------------------
A
Project Report
on
Robustifying Language Models with
Curriculum-Based Noise Injection
Submitted to
BHILAI INSTITUTE OF TECHNOLOGY, DURG
Affiliated to
CHHATTISGARH SWAMI VIVEKANAND TECHNICAL
UNIVERSITY
BHILAI
in partial fulfillment of
Bachelor of Technology
in
Computer Science and Engineering
by
Ayush Pradhan

6th Semester 

Aary

In [ ]:
# Middle of document — check if content looks clean
mid = len(cleaned_text) // 2
print("MIDDLE SECTION")
print("-" * 40)
print(cleaned_text[mid:mid + 500])

# Last 500 characters
print("\n\nEND OF DOCUMENT")
print("-" * 40)
print(cleaned_text[-500:])

MIDDLE SECTION
----------------------------------------
ce in highly complex, multi-step logic generation (such as ad-
vanced nested SQL joins or complex Pandas aggregations).

CHAPTER 4
Methodology
4.1
System Architecture Overview
The methodology of this research is structured as a continuous, end-to-end Machine Learn-
ing Operations (MLOps) pipeline. It transitions from synthetic adversarial data generation to
hardware-efficient model fine-tuning, culminating in empirical evaluation. The pipeline con-
sists of four primary modules:
Pristine
Instruc


END OF DOCUMENT
----------------------------------------
Learning
Representations, 2022.
[6] A. Vaswani, N. Shazeer, N. Parmar, J. Uszkoreit, L. Jones, A. N. Gomez, . Kaiser, and
I. Polosukhin, Attention is all you need, in Advances in neural information processing
systems, vol. 30, 2017.
[7] C. E. Shannon, A mathematical theory of communication, The Bell system technical
journal, vol. 27, no. 3, pp. 379423, 1948.
[8] S. Kullback and R. 

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/cleaned_text.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print(f"✅ Cleaned text saved to {output_path}")

✅ Cleaned text saved to ../data/processed/cleaned_text.txt
